# Market Dollar Bars

## Process the Data

- **Purpose:** Aggregate observed AAPL trades into activity-based dollar bars.
- **Settings:** Estimate one fixed threshold from median daily dollar value with `target_minutes=1` and `session_minutes=390`.
- **Data:** Observed AAPL trades produce the cached OHLCV dollar-bar artifact.
- **Decision:** Target about 390 bars on a median-volume session and reuse the cached artifact when present.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.preprocessing.market_structured_bars import (
    estimate_dollar_bar_threshold,
    get_dollar_bars,
    save_structured_bar_result,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
market_path = PROJECT_ROOT / "data/research_data/market/data/aapl_2025-01-01_2025-12-31.parquet"
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_bar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
if not dollar_bar_path.is_file():
    observations = pd.read_parquet(market_path)
    threshold = estimate_dollar_bar_threshold(
        observations,
        target_minutes=1,
        session_minutes=390,
    )
    result = get_dollar_bars(observations, threshold=threshold)
    save_structured_bar_result(result, dollar_bar_path)
dollar_bars = pd.read_parquet(dollar_bar_path)
dollar_bar_path

Matplotlib is building the font cache; this may take a moment.


PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/market/features/aapl_dollar_bar_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- **Purpose:** Inspect the bar schema, AAPL-only coverage, and OHLCV and notional ranges.
- **Settings:** No analytical parameters; read-only inspection.
- **Data:** Inspect the cached AAPL dollar-bar artifact.
- **Decision:** Do not rebuild bars or modify the cached artifact.

In [2]:
dollar_bars.head()

,end,start,symbol,open,high,low,close,volume,dollar_value,ticks,buy_volume,sell_volume
0,2025-01-02 14:30:01.548213+00:00,2025-01-02 13:45:58.902948+00:00,AAPL,250.000,250.00,248.785,248.820,3019.0,751454.90,42,1385.0,1634.0
1,2025-01-02 14:30:02.782298+00:00,2025-01-02 14:30:01.548237+00:00,AAPL,248.820,248.83,248.590,248.660,3067.0,762749.28,62,1627.0,1440.0
2,2025-01-02 14:30:04.858845+00:00,2025-01-02 14:30:02.838161+00:00,AAPL,248.590,248.71,248.560,248.570,3023.0,751586.88,37,1013.0,2010.0
3,2025-01-02 14:30:09.217978+00:00,2025-01-02 14:30:04.900185+00:00,AAPL,248.590,248.84,248.530,248.815,3061.0,761111.21,38,1762.0,1299.0
4,2025-01-02 14:30:10.290178+00:00,2025-01-02 14:30:09.443442+00:00,AAPL,248.805,248.85,248.805,248.845,3075.0,765166.74,22,1331.0,1744.0


In [3]:
dollar_bars.info()

<class 'pandas.DataFrame'>
RangeIndex: 106690 entries, 0 to 106689
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   end           106690 non-null  datetime64[us, UTC]
 1   start         106690 non-null  datetime64[us, UTC]
 2   symbol        106690 non-null  str                
 3   open          106690 non-null  float64            
 4   high          106690 non-null  float64            
 5   low           106690 non-null  float64            
 6   close         106690 non-null  float64            
 7   volume        106690 non-null  float64            
 8   dollar_value  106690 non-null  float64            
 9   ticks         106690 non-null  int64              
 10  buy_volume    106690 non-null  float64            
 11  sell_volume   106690 non-null  float64            
dtypes: datetime64[us, UTC](2), float64(8), int64(1), str(1)
memory usage: 10.2 MB


In [4]:
dollar_bars["symbol"].value_counts(dropna=False)

symbol
AAPL    106690
Name: count, dtype: int64

In [5]:
dollar_bars.describe()

,open,high,low,close,volume,dollar_value,ticks,buy_volume,sell_volume
count,106690.000000,106690.000000,106690.000000,106690.000000,106690.000000,1.066900e+05,106690.000000,106690.000000,106690.000000
mean,233.320116,233.419387,233.219750,233.320317,3373.292502,7.757699e+05,41.845815,1716.866407,1656.426094
std,28.199449,28.183560,28.215365,28.199545,811.700409,1.660012e+05,13.683890,800.511730,827.909514
min,169.300000,169.540000,168.620000,169.290000,2590.000000,7.454215e+05,1.000000,0.000000,0.000000
25%,209.590000,209.710000,209.495000,209.590000,2967.000000,7.510624e+05,33.000000,1290.000000,1225.000000
50%,231.702500,231.810000,231.605000,231.710000,3295.000000,7.579406e+05,41.000000,1671.000000,1610.000000
75%,257.208750,257.305000,257.090000,257.208750,3663.000000,7.662175e+05,50.000000,2074.000000,2018.000000
max,288.360000,288.610000,288.250000,288.350000,81088.000000,2.075573e+07,350.000000,49685.000000,79263.000000


In [6]:
dollar_bars.hist(bins=50, figsize=(14, 10))
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_46107/2908202076.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
